# E3 — Entrenamiento PCN v3

Entrenamiento desde cero con hiperparámetros corregidos:
- **400 épocas** (v1 = 100, v2 = 300 con bug de loss)
- **LR decay cada 100 épocas** (v1 era cada 40 → decaía demasiado rápido)
- **w_coarse = 0.5** (igual que v1, para que las métricas sean comparables)
- **Batch 64** — T4 lo aguanta bien

⏱️ **Tiempo estimado en T4: ~3.5 horas**

---
### Antes de ejecutar:
Menú → **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**

---
### Celdas opcionales:
- **Celda 7**: evalúa el modelo v2 (`epoch_300.pt`) antes de entrenar — útil para comparar
- **Celda 9**: cambia `--epochs 400` por menos épocas para un test rápido

In [ ]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELDA 2: Clonar repo e instalar dependencias ───────────────
# El repo es privado → necesitas un token de GitHub.
#
# Cómo crear el token (1 sola vez):
#   GitHub → tu foto → Settings → Developer settings
#   → Personal access tokens → Tokens (classic) → Generate new token
#   → Selecciona solo 'repo' → Generate → copia el token (empieza por ghp_)

import os
from getpass import getpass

REPO_DIR = '/content/TFM'

if not os.path.exists(REPO_DIR):
    token = getpass('Pega tu token de GitHub (ghp_...) y pulsa Enter: ')
    repo_url = f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D'
    os.system(f'git clone {repo_url} {REPO_DIR}')
    del token
else:
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
os.system('git checkout raquel/e3')
print('Directorio de trabajo:', os.getcwd())

import subprocess
subprocess.run(['pip', 'install', 'torch', 'numpy', 'matplotlib', '--quiet'])
print('Dependencias instaladas.')

In [ ]:
# ── CELDA 3: RUTAS DE DRIVE ────────────────────────────────────
# Si quieres cambiar a otro Drive, solo cambia DRIVE.
# Los nombres de carpeta siguen la estructura acordada:
#   Datos_E2_E3/E3/modelos/<version>/   ← checkpoints .pt
#   Datos_E2_E3/E3/resultados/<version>/ ← métricas + figuras

DRIVE   = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3'

# Versión de este entrenamiento (cambia aquí si lanzas otro experimento)
VERSION = 'v3_pcn'

# Datos de entrenamiento
RUTA_SINTETICO    = f'{DRIVE}/Datos_E2_E3/General/sintetico_roturas'
RUTA_FB_PROCESADO = f'{DRIVE}/Datos_E2_E3/General/Fantastik_Break_Preprocesado'

# Modelo de referencia v2 (para evaluar antes de entrenar v3)
RUTA_MODELO_V2 = f'{BASE_E3}/modelos/v2_pcn/epoch_300.pt'

# Dónde guardar el modelo y resultados de esta versión
RUTA_SALIDA_MODELO     = f'{BASE_E3}/modelos/{VERSION}'
RUTA_SALIDA_RESULTADOS = f'{BASE_E3}/resultados/{VERSION}'

print('Rutas configuradas:')
print(f'  sintetico       : {RUTA_SINTETICO}')
print(f'  fantastic_breaks: {RUTA_FB_PROCESADO}')
print(f'  modelo v2       : {RUTA_MODELO_V2}')
print(f'  salida modelo   : {RUTA_SALIDA_MODELO}')
print(f'  salida resultados: {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 4: Verificar que las rutas existen ───────────────────
from pathlib import Path

rutas = {
    'sintetico_roturas'      : RUTA_SINTETICO,
    'fantastic_breaks'       : RUTA_FB_PROCESADO,
    'modelo_v2 (epoch_300)'  : RUTA_MODELO_V2,
}

ok = True
for nombre, ruta in rutas.items():
    existe = Path(ruta).exists()
    emoji  = '✅' if existe else '❌'
    print(f'  {emoji} {nombre}: {ruta}')
    if not existe:
        ok = False

if ok:
    print('\nTodo encontrado. Puedes continuar.')
else:
    print('\n⚠️  Alguna ruta no existe. Revisa el Drive antes de continuar.')

In [ ]:
# ── CELDA 5: Copiar datos desde Drive ─────────────────────────
import subprocess
from pathlib import Path

def copiar_dir(src, dst):
    """Copia carpeta de Drive evitando bucles de shortcuts/symlinks."""
    src, dst = Path(src), Path(dst)
    if dst.exists() and any(dst.glob('*.npy')):
        n = len(list(dst.glob('*.npy')))
        print(f'  [OK] ya existe: {dst.name}  ({n} .npy)')
        return
    if not src.exists():
        print(f'  [ERROR] no encontrado: {src}')
        return
    dst.mkdir(parents=True, exist_ok=True)
    print(f'  Copiando {src.name} (puede tardar varios minutos)...', flush=True)
    r = subprocess.run(['rsync', '-a', '--no-links', f'{src}/', str(dst)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  [ERROR rsync] {r.stderr[:300]}')
    else:
        n = len(list(dst.glob('*.npy')))
        print(f'  listo — {n} archivos .npy copiados.')

copiar_dir(RUTA_SINTETICO,    'Datos/sintetico/roturas')
copiar_dir(RUTA_FB_PROCESADO, 'Datos/fantastic_breaks/procesado')

print()
for c in ['Datos/sintetico/roturas', 'Datos/fantastic_breaks/procesado']:
    n = len(list(Path(c).glob('*.npy'))) if Path(c).exists() else 0
    print(f'  {c}: {n} .npy')

In [ ]:
# ── CELDA 6: Verificar GPU ─────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  Sin GPU. Ve a Entorno de ejecución → Cambiar tipo → T4 GPU')

In [ ]:
# ── CELDA 7 [OPCIONAL]: Evaluar modelo v2 antes de entrenar v3 ─
# Ejecuta esto primero para ver si epoch_300.pt mejoró sobre v1 (CD=0.077).
# Si el CD baja → v2 es mejor que v1. Si no → v3 parte de cero igualmente.

import shutil
from pathlib import Path

# Copia epoch_300.pt de Drive al directorio de trabajo
Path('E3/checkpoints').mkdir(parents=True, exist_ok=True)
shutil.copy2(RUTA_MODELO_V2, 'E3/checkpoints/best.pt')
print(f'Cargado: {RUTA_MODELO_V2}')

!python -m E3.evaluate \
    --checkpoint E3/checkpoints/best.pt \
    --salida     E3/resultados_v2

# Guardar resultados v2 en Drive
import shutil, time
RUTA_SALIDA_V2 = f'{BASE_E3}/resultados/v2_pcn'
Path(RUTA_SALIDA_V2).mkdir(parents=True, exist_ok=True)
shutil.copytree('E3/resultados_v2', RUTA_SALIDA_V2, dirs_exist_ok=True)
print(f'Resultados v2 guardados en Drive: {RUTA_SALIDA_V2}')

In [ ]:
# ── CELDA 8: ENTRENAR v3 (desde cero) ─────────────────────────
# Deja esta celda corriendo (~3.5 horas en T4).
# El progreso aparece línea a línea.
#
# Mejoras respecto a v1 y v2:
#   --epochs 400     → más épocas para que converja bien
#   --lr_decay 100   → decay más lento (v1 era 40 → LR caía demasiado rápido)
#   --w_coarse 0.5   → igual que v1, métricas comparables
#   sin --resume     → entrenamiento limpio desde cero

!python -m E3.train \
    --epochs     400 \
    --lr         1e-4 \
    --lr_decay   100 \
    --batch_size 64 \
    --w_coarse   0.5

In [ ]:
# ── CELDA 9: Guardar modelo en Drive ──────────────────────────
import shutil
from pathlib import Path

Path(RUTA_SALIDA_MODELO).mkdir(parents=True, exist_ok=True)

shutil.copy2('E3/checkpoints/best.pt', f'{RUTA_SALIDA_MODELO}/best.pt')
print(f'Modelo v3 guardado: {RUTA_SALIDA_MODELO}/best.pt')

# También guarda los checkpoints periódicos de las últimas épocas
for ckpt in sorted(Path('E3/checkpoints').glob('epoch_3??.pt')):
    dst = Path(RUTA_SALIDA_MODELO) / ckpt.name
    shutil.copy2(ckpt, dst)
    print(f'  + {ckpt.name}')

In [ ]:
# ── CELDA 10: Evaluar modelo v3 ────────────────────────────────
# Compara el resultado con v1 (CD=0.077, F-Score=0.019).

!python -m E3.evaluate \
    --checkpoint E3/checkpoints/best.pt \
    --salida     E3/resultados

# Guardar resultados en Drive
import shutil
from pathlib import Path

Path(RUTA_SALIDA_RESULTADOS).mkdir(parents=True, exist_ok=True)
shutil.copytree('E3/resultados', RUTA_SALIDA_RESULTADOS, dirs_exist_ok=True)
print(f'Resultados v3 guardados en Drive: {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 11: Ver figuras inline ────────────────────────────────
from IPython.display import Image, display
from pathlib import Path

figuras = sorted(Path('E3/resultados').glob('figura_*.png'))
if not figuras:
    print('No hay figuras. Ejecuta primero la Celda 10.')
else:
    mejores = [f for f in figuras if 'mejor' in f.name]
    peores  = [f for f in figuras if 'peor'  in f.name]

    print(f'=== MEJORES ({len(mejores)}) ===')
    for f in mejores:
        print(f.name)
        display(Image(str(f), width=1000))

    print(f'\n=== PEORES ({len(peores)}) ===')
    for f in peores:
        print(f.name)
        display(Image(str(f), width=1000))